In [3]:
import csv
from pathlib import Path

INPUT_CSV = "domain_prices.csv"
OUTPUT_CSV = "domains_for_sale.csv"

def is_numeric_price(value: str) -> bool:
    if not value:
        return False
    v = value.strip()
    if v.upper() in {"NONE", "NO_SITE", "NO_TEXT"}:
        return False
    try:
        float(v)
        return True
    except ValueError:
        return False

input_path = Path(INPUT_CSV)
if not input_path.is_file():
    raise FileNotFoundError(f"{INPUT_CSV} not found")

with open(INPUT_CSV, newline="", encoding="utf-8") as fin, \
     open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as fout:

    reader = csv.DictReader(fin)
    fieldnames = reader.fieldnames
    writer = csv.DictWriter(fout, fieldnames=fieldnames)
    writer.writeheader()

    count = 0
    for row in reader:
        if is_numeric_price(row["price"]):
            writer.writerow(row)
            count += 1

print(f"Wrote {count} domains with numeric prices to {OUTPUT_CSV}")


Wrote 127 domains with numeric prices to domains_for_sale.csv


In [6]:
import csv
import json
import time
import re
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from openai import OpenAI

# ---------- CONFIG ----------

INPUT_CSV = "domains_for_sale.csv"          # your existing file
OUTPUT_CSV = "domains_for_sale_usd.csv"     # refined output

OPENAI_SLEEP_SECONDS = 0.3                  # be nice to rate limits
MAX_TEXT_CHARS = 8000                       # cap text we send to GPT

client = OpenAI()


# ---------- helpers to fetch & clean page ----------

def fetch_page(url: str):
    """
    Fetch a URL and return (final_url, html_text) or (None, None).
    """
    headers = {
        "User-Agent": "DomainSaleChecker/1.0 (+https://example.com/contact)"
    }

    try:
        resp = requests.get(url, timeout=15, headers=headers, allow_redirects=True)
        if resp.status_code == 200 and "text/html" in resp.headers.get("Content-Type", ""):
            return resp.url, resp.text
    except requests.RequestException:
        return None, None

    return None, None


def extract_visible_text(html: str) -> str:
    """
    Strip scripts/styles/etc and return a cleaned text string.
    """
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript", "iframe", "header", "footer"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# ---------- GPT analysis ----------

def analyze_for_sale(page_text: str, domain: str, original_price: str | float):
    """
    Use GPT to decide if the domain itself is for sale, and get price + USD conversion.

    Returns a dict:
      {
        "is_for_sale": bool,
        "price_original": float | None,
        "currency": str | None,
        "price_usd": float | None
      }
    """
    # truncate to control cost
    page_text = page_text[:MAX_TEXT_CHARS]

    prompt = f"""
You are analyzing a web page to see if the DOMAIN NAME itself is for sale.

The page text may be in any language.

You are given:
- Domain: "{domain}"
- A previously extracted numeric value: "{original_price}" (this may or may not be the real sale price of the domain).

Your tasks:

1. Decide if the domain/website "{domain}" is clearly being offered for sale, lease, or auction.
   - This includes phrases like "this domain is for sale", "buy this domain",
     "acquista questo dominio", "zu verkaufen", "dominio en venta", etc.
   - Do NOT mark it as for sale if the page is just selling products or services,
     not the domain/website itself.

2. If the domain is for sale and a price is shown:
   - Extract the main sale price for the domain.
   - Detect the currency (e.g. USD, EUR, CNY, JPY, GBP, etc.).
   - Convert the price to USD using your best estimate of the exchange rate.

3. If there is no clear sale price for the domain, or you are not sure, set price_original and price_usd to null.

Return ONLY a JSON object with this exact structure:

{{
  "is_for_sale": true or false,
  "price_original": number or null,
  "currency": string or null,
  "price_usd": number or null
}}

Do not include any extra text outside the JSON.

Page text:
\"\"\"{page_text}\"\"\""""

    response = client.responses.create(
        model="gpt-5-nano",  # or gpt-4.1 / gpt-4.1-nano depending on cost/speed
        input=prompt,
    )

    raw = response.output_text
    # print("RAW GPT:", raw)  # uncomment for debugging

    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        return {
            "is_for_sale": False,
            "price_original": None,
            "currency": None,
            "price_usd": None,
        }

    is_for_sale = bool(data.get("is_for_sale", False))

    def to_float_or_none(x):
        if x is None:
            return None
        try:
            return float(x)
        except (TypeError, ValueError):
            return None

    price_original = to_float_or_none(data.get("price_original"))
    price_usd = to_float_or_none(data.get("price_usd"))
    currency = data.get("currency")
    if currency is not None:
        currency = str(currency).upper()

    return {
        "is_for_sale": is_for_sale,
        "price_original": price_original,
        "currency": currency,
        "price_usd": price_usd,
    }


# ---------- main pass over domains_for_sale.csv ----------

input_path = Path(INPUT_CSV)
if not input_path.is_file():
    raise FileNotFoundError(f"{INPUT_CSV} not found")

with open(INPUT_CSV, newline="", encoding="utf-8") as fin, \
     open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as fout:

    reader = csv.DictReader(fin)

    # New schema with normalized USD
    fieldnames = [
        "domain",
        "url",
        "price_original",
        "currency",
        "price_usd",
    ]
    writer = csv.DictWriter(fout, fieldnames=fieldnames)
    writer.writeheader()

    count = 0
    total = 0

    for row in reader:
        domain = row.get("domain", "").strip()
        url = row.get("url", "").strip()
        original_price = row.get("price", "").strip()

        total += 1
        if not url:
            # Shouldn't happen in domains_for_sale.csv, but be safe
            continue

        print(f"[{total}] Re-checking {domain} ({url}) ...")

        final_url, html = fetch_page(url)
        if not final_url or not html:
            print("   -> Could not refetch page.")
            continue

        text = extract_visible_text(html)
        if not text:
            print("   -> No text found on page.")
            continue

        info = analyze_for_sale(text, domain, original_price)

        if info["is_for_sale"] and info["price_usd"] is not None:
            writer.writerow({
                "domain": domain,
                "url": final_url,
                "price_original": info["price_original"],
                "currency": info["currency"],
                "price_usd": info["price_usd"],
            })
            count += 1
            print(f"   -> FOR SALE: {info['price_original']} {info['currency']} (~{info['price_usd']} USD)")
        else:
            print("   -> Not clearly for sale, or no usable price.")

        fout.flush()
        time.sleep(OPENAI_SLEEP_SECONDS)

print(f"Finished. Wrote {count} confirmed for-sale domains with USD prices to {OUTPUT_CSV}")


[1] Re-checking a.ar (https://a.ar) ...
   -> FOR SALE: 100000.0 USD (~100000.0 USD)
[2] Re-checking b.ml (https://b.ml) ...
   -> FOR SALE: 80000.0 USD (~80000.0 USD)
[3] Re-checking c.cm (https://c.cm) ...
   -> FOR SALE: 270000.0 USD (~270000.0 USD)
[4] Re-checking c.la (http://c.la) ...
   -> Not clearly for sale, or no usable price.
[5] Re-checking c.td (http://c.td) ...
   -> Not clearly for sale, or no usable price.
[6] Re-checking d.am (https://d.am) ...
   -> FOR SALE: 199999.0 USD (~199999.0 USD)
[7] Re-checking d.bw (http://d.bw) ...
   -> Not clearly for sale, or no usable price.
[8] Re-checking d.cl (https://d.cl) ...
   -> FOR SALE: 10000.0 EUR (~11000.0 USD)
[9] Re-checking d.my (https://d.my) ...
   -> Not clearly for sale, or no usable price.
[10] Re-checking d.nf (https://d.nf) ...
   -> Not clearly for sale, or no usable price.
[11] Re-checking d.vu (http://d.vu) ...
   -> Not clearly for sale, or no usable price.
[12] Re-checking e.cv (https://e.cv) ...
   -> FOR SA